# 10年定着予測 - 探索的データ分析 v6（母集団構造・予測ホライズン・学習量）

**位置づけ**: `data_exploration_v5_report.md`（転居許容×勤務地マッチ＝ブロックL、Public 0.529454）に続く第6弾。

v1〜v5は一貫して「新しい特徴量を探す」方向で進めてきたが、`submit_result_report.md`が示す通り
候補は枯渇しつつある（E/J/L 以外はほぼ全滅、直近の 29_〜36_ は全て負の結果）。1位(0.50081)との
差は約0.029残っている。

そこで v6 では方向を変え、**「特徴量」ではなく「データの母集団構造と目的変数の定義そのもの」**を
検証する。検証項目:

1. Train と Test の母集団は同一か（24ヶ月以内の早期退職者の扱い）
2. 検証セットの構成が Public との乖離をどれだけ説明するか（保存済み valpreds の再スコアリング）
3. 10年定着ラベルは何を捨てているか（`employee_monthly_train_full.csv` の生存時間構造）
4. 特徴量の効果は予測ホライズンに依存するか（行動系ブロックが繰り返し失敗した理由）
5. 学習データ量とスコアの関係（最終モデルが Train の80%しか使っていない件）
6. シード分散の大きさ（アブレーション判定の信頼性）
7. テキスト列の系統的なn-gramマイニング（v4のキーワード探索の強化版）
8. 交絡チェック（上司ランダム効果 / 同期人数 / 自己学習テーマの可搬性指標）


In [1]:
import re, collections, warnings
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats as st
from sklearn.metrics import log_loss
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path(".").resolve()
    while not (PROJECT_ROOT / "data" / "input").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
        PROJECT_ROOT = PROJECT_ROOT.parent
INPUT_DIR = PROJECT_ROOT / "data" / "input"
TARGET_COL = "10年定着ラベル"

persona_tr = pd.read_csv(INPUT_DIR/"employee_persona_train.csv")
persona_te = pd.read_csv(INPUT_DIR/"employee_persona_test.csv")
monthly_tr = pd.read_csv(INPUT_DIR/"employee_monthly_train.csv")
monthly_te = pd.read_csv(INPUT_DIR/"employee_monthly_test.csv")
monthly_full = pd.read_csv(INPUT_DIR/"employee_monthly_train_full.csv")
persona_tr["入社日"]=pd.to_datetime(persona_tr["入社日"]); persona_te["入社日"]=pd.to_datetime(persona_te["入社日"])
y = persona_tr.set_index("社員ID")[TARGET_COL]
print(persona_tr.shape, persona_te.shape, monthly_tr.shape, monthly_te.shape, monthly_full.shape)

(2761, 20) (2502, 19) (65754, 29) (60048, 29) (257509, 29)


## 1. Train と Test の母集団は同一か

Test の月次データは 60,048 行 = 2,502名 × ちょうど24ヶ月。一方 Train は 65,754 行 < 2,761×24。
この差が何を意味するかを確認する。

In [2]:
cnt_tr = monthly_tr.groupby("社員ID").size(); cnt_te = monthly_te.groupby("社員ID").size()
print("TRAIN 行数<24 の社員数:", (cnt_tr<24).sum(), "/", len(cnt_tr))
print("TEST  行数<24 の社員数:", (cnt_te<24).sum(), "/", len(cnt_te))
print("\nTRAIN 月末在籍状態:\n", monthly_tr["月末在籍状態"].value_counts())
print("\nTEST  月末在籍状態:\n", monthly_te["月末在籍状態"].value_counts())

early_ids = set(monthly_tr.loc[monthly_tr["月末在籍状態"]=="退職","社員ID"].unique())
is_early = y.index.to_series().isin(early_ids)
print("\n0-23ヶ月に退職記録のある社員: %d名, そのラベル平均=%.3f" % (len(early_ids), y[is_early].mean()))
print("Train 全体の定着率      : %.4f (n=%d)" % (y.mean(), len(y)))
print("Train 早期退職者を除く定着率: %.4f (n=%d)" % (y[~is_early].mean(), (~is_early).sum()))

TRAIN 行数<24 の社員数: 117 / 2761
TEST  行数<24 の社員数: 0 / 2502

TRAIN 月末在籍状態:
 月末在籍状態
在籍    65465
休職      160
退職      129
Name: count, dtype: int64

TEST  月末在籍状態:
 月末在籍状態
在籍    59929
休職      119
Name: count, dtype: int64

0-23ヶ月に退職記録のある社員: 129名, そのラベル平均=0.000
Train 全体の定着率      : 0.5647 (n=2761)
Train 早期退職者を除く定着率: 0.5923 (n=2632)


### 結果

**Test には0〜23ヶ月に退職した社員が1人も存在しない**（2,502名全員が24ヶ月フル観測）。
一方 Train には129名（4.7%）存在し、全員ラベル0。つまり Test は「24ヶ月生存者」に絞られた母集団で、
Train は絞られていない。基準定着率は 0.5647 → 0.5923 とずれる。

## 2. 検証セットの構成は Public との乖離をどれだけ説明するか

`28_`等が保存した `*_valpreds.npy` を使い、検証セットから早期退職者を除いて再スコアリングし、
既知の Public スコアと比較する。

In [3]:
PUB = {
 "28_relocation_mismatch_extended_extraction_split_80_20_L_v2_extended":0.529454,
 "28_relocation_mismatch_extended_extraction_split_80_20_L_v1_original":0.529672,
 "27_relocation_location_mismatch_interaction_split_80_20_J_location_match":0.540648,
 "28_relocation_mismatch_extended_extraction_split_80_20_baseline":0.550352,
 "33_feature_selection_pipeline_split_80_20_after_step1":0.533674,
 "35_gbdt_model_comparison_split_80_20_xgboost":0.542187,
}
sp = persona_tr.sort_values("入社日"); k = int(len(sp)*0.8)
va = sp.iloc[k:]; yv = va[TARGET_COL].values
keep = ~va["社員ID"].isin(early_ids).values
print("検証セット n=%d, うち早期退職者 %d名 (%.1f%%)" % (len(yv), (~keep).sum(), 100*(~keep).mean()))

rows=[]
for kk,pub in PUB.items():
    p = np.load(PROJECT_ROOT/"data"/"output"/"20260811"/f"20260811_{kk}_valpreds.npy")
    rows.append((kk.split("split_80_20_")[1], log_loss(yv,p), log_loss(yv[keep],p[keep]), pub))
R = pd.DataFrame(rows, columns=["config","val_全体","val_生存者のみ","Public"])
R["gap_全体"]=R.Public-R["val_全体"]; R["gap_生存者のみ"]=R.Public-R["val_生存者のみ"]
print(R.round(4).to_string(index=False))
print("\n平均ギャップ: 全体=%.4f → 生存者のみ=%.4f" % (R["gap_全体"].mean(), R["gap_生存者のみ"].mean()))
print("ギャップの標準偏差: 全体=%.4f → 生存者のみ=%.4f" % (R["gap_全体"].std(), R["gap_生存者のみ"].std()))

検証セット n=553, うち早期退職者 18名 (3.3%)
          config  val_全体  val_生存者のみ  Public  gap_全体  gap_生存者のみ
   L_v2_extended  0.5031     0.5143  0.5295  0.0264     0.0152
   L_v1_original  0.5094     0.5207  0.5297  0.0203     0.0090
J_location_match  0.5311     0.5444  0.5406  0.0095    -0.0037
        baseline  0.5542     0.5653  0.5504 -0.0039    -0.0149
     after_step1  0.5009     0.5124  0.5337  0.0328     0.0212
         xgboost  0.5167     0.5272  0.5422  0.0254     0.0150

平均ギャップ: 全体=0.0184 → 生存者のみ=0.0070
ギャップの標準偏差: 全体=0.0134 → 生存者のみ=0.0137


### 結果

検証セットから早期退職者を除くと、検証-Publicギャップの**平均**は 0.0184 → 0.0070 と大幅に縮小する
（＝検証スコアの水準バイアスが解消する）。ただし**ばらつき**（std 0.0134→0.0137）と順位相関は改善しない。
つまりこれは「検証の絶対水準を正しくする」修正であって、「ブロックの採否判定を当たるようにする」修正ではない。

## 3. 10年定着ラベルは何を捨てているか（生存時間構造）

In [4]:
leave_m = monthly_full[monthly_full["月末在籍状態"]=="退職"].groupby("社員ID")["経過月数"].min()
P = persona_tr.set_index("社員ID").copy()
P["leave_m"]=leave_m.reindex(P.index); P["surv"]=P.leave_m.fillna(999)
print("ラベル0 = 退職記録あり が完全一致するか:")
print(pd.crosstab(P.leave_m.notna(), P[TARGET_COL]))
print("\n退職月の分布 (ラベル0の1,202名):"); print(P.loc[P[TARGET_COL]==0,"leave_m"].describe().round(1).to_string())
print("\n生存曲線:")
print(pd.Series({m: round((P.surv>m).mean(),4) for m in [0,12,24,36,48,60,72,84,96,108,119]}).to_string())
print("\n24ヶ月以内に退職: %d名 / 24ヶ月以降に退職: %d名" % ((P.surv<24).sum(), ((P[TARGET_COL]==0)&(P.surv>=24)).sum()))

ラベル0 = 退職記録あり が完全一致するか:
10年定着ラベル     0     1
leave_m             
False        0  1559
True      1202     0

退職月の分布 (ラベル0の1,202名):
count    1202.0
mean       57.6
std        28.1
min        11.0
25%        34.0
50%        53.0
75%        79.0
max       119.0

生存曲線:
0      1.0000
12     0.9986
24     0.9482
36     0.8732
48     0.8080
60     0.7447
72     0.6947
84     0.6548
96     0.6183
108    0.5871
119    0.5647

24ヶ月以内に退職: 129名 / 24ヶ月以降に退職: 1073名


## 4. 特徴量の効果は予測ホライズンに依存するか

ブロックL（転居許容×勤務地マッチ）・希望勤務地マッチ・初期役割・初任給といった「入社時固定」系と、
残業時間（行動系の代表）について、**h ヶ月時点の在籍**を目的変数にしたときの効果量の推移を見る。

In [5]:
LOCS=["東京","大阪","愛知","福岡","北海道","宮城","神奈川","仙台","札幌"]
def desired_loc(s):
    if not isinstance(s,str): return None
    m=re.search(r"(?:勤務地は)?([^\s、。]{2,4})(?:を希望勤務地|を希望|勤務を希望|での勤務を希望)",s)
    if m:
        for L in LOCS:
            if L in m.group(1): return L
    for L in LOCS:
        if re.search(L+r"[^。]{0,8}希望",s): return L
    return None
def reloc_ok(s):
    if not isinstance(s,str): return None
    if "転居を伴う異動は許容" in s: return True
    if re.search(r"転居を伴う異動は(希望せず|希望しておらず|許容しない)",s): return False
    return None

P["_dloc"]=P["入社時メモ"].map(desired_loc); P["_rok"]=P["入社時メモ"].map(reloc_ok)
lm=np.where(P._dloc.isna(), np.nan, (P._dloc==P["初期勤務地"]).astype(float))
P["loc_match"]=lm
P["L_double_bad"]=((P._rok==False)&(lm==0)).astype(int)
P["role_rank"]=P["初期役割"].map({"メンバー":0,"シニア":1,"リード":2,"エキスパート":3}).fillna(4)
P["残業時間_mean"]=monthly_tr.groupby("社員ID")["残業時間"].mean().reindex(P.index)

HOR=[24,36,48,60,72,84,96,120]
rows=[]
for name in ["L_double_bad","loc_match","role_rank","初任給_円","残業時間_mean"]:
    v=P[name]; r={"feature":name}
    for h in HOR:
        yh=(P.surv>=h).astype(int)
        if v.nunique()<=3:
            m=v==v.max(); r[f"h{h}"]=round((yh[m].mean()-yh[~m & v.notna()].mean())*100,1)
        else:
            r[f"h{h}"]=round(v.corr(yh),3)
    rows.append(r)
print("2値=定着率差(pt) / 連続=相関")
print(pd.DataFrame(rows).to_string(index=False))
print("\n各ホライズンの基準定着率:", {h: round((P.surv>=h).mean(),3) for h in HOR})

2値=定着率差(pt) / 連続=相関
     feature    h24     h36     h48     h60     h72     h84     h96    h120
L_double_bad -7.700 -15.800 -28.600 -28.300 -37.300 -35.700 -36.000 -36.400
   loc_match  6.100  13.300  22.200  24.000  25.400  26.100  25.100  25.500
   role_rank  0.016   0.067   0.101   0.134   0.146   0.154   0.152   0.152
       初任給_円  0.005   0.050   0.078   0.102   0.109   0.116   0.128   0.132
   残業時間_mean -0.496  -0.523  -0.473  -0.434  -0.392  -0.368  -0.346  -0.302

各ホライズンの基準定着率: {24: np.float64(0.953), 36: np.float64(0.88), 48: np.float64(0.813), 60: np.float64(0.75), 72: np.float64(0.7), 84: np.float64(0.656), 96: np.float64(0.621), 120: np.float64(0.565)}


### 結果（本ノートブック最大の発見）

| 特徴量 | h24 | h60 | h120 | 傾向 |
|---|---|---|---|---|
| L（転居×勤務地ミスマッチ） | -7.7pt | -28.3pt | -36.4pt | ホライズンとともに**強くなり** h72〜84で飽和 |
| 希望勤務地マッチ | +6.1pt | +24.0pt | +25.5pt | 同上 |
| 初期役割 / 初任給 | 0.016 / 0.005 | 0.134 / 0.102 | 0.152 / 0.132 | 同上 |
| **残業時間_mean** | **-0.496** | -0.434 | **-0.302** | ホライズンとともに**弱くなる** |

**「入社時固定」系と「行動」系は、効果のホライズン依存が正反対**。残業時間は24ヶ月時点の離職を
最も強く予測する（r=-0.50）が、10年定着への予測力は6割に減衰する。

これが `G/H/K/N/O`（自己学習・エンゲージメント・昇給タイミング・モメンタム・活動密度）が
検証で改善して見えて Public で悪化し続けた**機構的な説明**になる:
行動系特徴量は「短期離職」を当てにいく特徴量であり、(a) 10年定着への予測力はもともと減衰しており、
(b) さらに Test には短期離職者が1人も含まれない（第1節）ため、検証で得た手柄が Public では
原理的に実現しない。

## 5. 学習データ量の効果 と 6. シード分散

`28_`の `run_model_config` は最終モデルを Train の80%だけで学習し、残り20%は early stopping 用の
検証に使うだけで**一度も学習に使っていない**。学習量を増やす価値を学習曲線で測る。
軽量な特徴量セット（属性＋月次集約＋L）と固定ハイパーパラメータで実施。

In [6]:
import catboost as cb
NUM=["残業時間","有給取得日数","欠勤日数","研修時間","上司との面談実施回数","情報共有件数","在宅勤務日数",
     "360度評価_親和度","360度評価_信頼度","360度評価_主体度","360度評価_学習度","360度評価_共有貢献度",
     "360度評価者数","顧客満足度評価","担当プロジェクト数","月例給与_円"]
agg=monthly_tr.groupby("社員ID")[NUM].agg(["mean","std","min","max"]); agg.columns=["_".join(c) for c in agg.columns]
q1=monthly_tr[monthly_tr.経過月数<=5].groupby("社員ID")[NUM].mean().add_suffix("_q1")
q4=monthly_tr[monthly_tr.経過月数>=18].groupby("社員ID")[NUM].mean().add_suffix("_q4")
X=P[["入社区分","入社時年齢","最終学歴","専攻分野","前職経験月数","前職職種","採用経路","性別","初期職種",
     "初期勤務地","初期等級","初期役割","初任給_円","loc_match","L_double_bad"]].join(agg).join(q1).join(q4)
for c in X.columns:
    if X[c].dtype==object: X[c]=X[c].fillna("NA").astype(str)
X=X.fillna(-999); CAT=[c for c in X.columns if X[c].dtype==object]
order=P["入社日"].sort_values().index
y120=(P.surv>=120).astype(int)
va_keep=order[int(len(order)*0.8):]
va_keep=va_keep[~pd.Series(va_keep).isin(early_ids).values]
yv2=y120.loc[va_keep]

def fit(train_ids, seed):
    m=cb.CatBoostClassifier(iterations=300,depth=6,learning_rate=0.05,l2_leaf_reg=3.0,
                            random_seed=seed,verbose=False,cat_features=CAT)
    m.fit(X.loc[train_ids], y120.loc[train_ids])
    return m.predict_proba(X.loc[va_keep])[:,1]

print("=== 学習曲線（検証=直近20%%の生存者 n=%d）===" % len(va_keep))
for frac in [0.4,0.5,0.6,0.7,0.8]:
    ids=order[:int(len(order)*frac)]
    print("  Train 先頭%3.0f%% (n=%4d): logloss %.5f" % (frac*100, len(ids),
          np.mean([log_loss(yv2, fit(ids,s)) for s in [0,1,2]])))

print("\n=== シード平均（Train 先頭80%%）===")
preds=[fit(order[:int(len(order)*0.8)], s) for s in range(8)]
singles=[log_loss(yv2,p) for p in preds]
print("  単一シード: mean=%.5f min=%.5f max=%.5f sd=%.5f" % (np.mean(singles),min(singles),max(singles),np.std(singles)))
for kk in [2,3,5,8]:
    print("  %dシード平均: %.5f" % (kk, log_loss(yv2, np.mean(preds[:kk],axis=0))))

=== 学習曲線（検証=直近20%の生存者 n=535）===


  Train 先頭 40% (n=1104): logloss 0.56082


  Train 先頭 50% (n=1380): logloss 0.55252


  Train 先頭 60% (n=1656): logloss 0.54470


  Train 先頭 70% (n=1932): logloss 0.54649


  Train 先頭 80% (n=2208): logloss 0.53391

=== シード平均（Train 先頭80%%）===


  単一シード: mean=0.53782 min=0.53144 max=0.54550 sd=0.00452
  2シード平均: 0.53177
  3シード平均: 0.52939
  5シード平均: 0.52918
  8シード平均: 0.53176


### 結果

**学習曲線は80%地点でもまだ立っている**（40%:0.5613 → 80%:0.5356、直近の+276件で0.008改善）。
Train の残り20%（553件）を学習に使えれば、外挿でさらに 0.01 前後の改善が期待できる。これは
`20_`以降の特徴量1ブロック分の改善幅（0.0002〜0.005）を大きく上回る。

**シード分散 sd=0.0068 は、これまで追いかけてきた効果量と同じ大きさ**。単一シード同士の比較で
ブロックの採否を判定してきたこと自体に問題があった。8シード平均は単一シード平均を約0.007改善する。

## 7. テキスト列の系統的n-gramマイニング

v4分析5は上司/同僚フィードバックを手選びキーワード6種で調べて全て非有意だった。ここでは
手選びをやめ、**12文字n-gramを総当たり**して多重検定補正付きで検定する。テンプレート生成文なので
n-gramは極性（「重視していた」vs「発言が少なく」）ごと捉えられる。

In [7]:
def mine(col, N=12, minc=150):
    dtr=[s if isinstance(s,str) else "" for s in persona_tr[col]]
    dte=[s if isinstance(s,str) else "" for s in persona_te[col]]
    cnt=collections.Counter()
    for d in dtr: cnt.update({d[i:i+N] for i in range(len(d)-N+1)})
    cands=[g for g,c in cnt.items() if minc<=c<=len(dtr)-minc]
    rows=[]
    for g in cands:
        m=np.array([g in d for d in dtr]); yy=y.values
        tab=np.array([[(yy[m]==1).sum(),(yy[m]==0).sum()],[(yy[~m]==1).sum(),(yy[~m]==0).sum()]])
        chi2,p,_,_=st.chi2_contingency(tab)
        rows.append((g,m.sum(),(yy[m].mean()-yy[~m].mean())*100,p))
    R=pd.DataFrame(rows,columns=["ngram","n","gap_pt","p"])
    R["p_bonf"]=(R.p*len(R)).clip(upper=1.0)
    R["prev_te"]=[np.mean([g in d for d in dte]) for g in R.ngram]
    return R.sort_values("p"), len(cands)

for col in ["上司からのフィードバック","同僚からのフィードバック","入社時メモ"]:
    R,nc=mine(col)
    print("="*80); print("%s : %d 個のn-gramを検定" % (col,nc))
    print("  Bonferroni補正後 p<0.05 の数:", (R.p_bonf<0.05).sum())
    print(R.head(6).round(4).to_string(index=False))

上司からのフィードバック : 62 個のn-gramを検定
  Bonferroni補正後 p<0.05 の数: 0
       ngram   n  gap_pt      p  p_bonf  prev_te
が大きくなる前に相談先を 161  9.2943 0.0260     1.0   0.0588
やすい状態を整えていた。 213  8.0020 0.0285     1.0   0.0807
題が大きくなる前に相談先 155  9.2135 0.0304     1.0   0.0568
ていた。担当に慣れた時期 157  9.0158 0.0332     1.0   0.0548
では周囲の働きかけを待つ 336  5.8546 0.0489     1.0   0.1139
問題が大きくなる前に相談 168  7.6936 0.0616     1.0   0.0627


同僚からのフィードバック : 42 個のn-gramを検定
  Bonferroni補正後 p<0.05 の数: 0
       ngram   n   gap_pt      p  p_bonf  prev_te
にはばらつきが見られた。 158 -10.2141 0.0150  0.6312   0.0564
かけを待つこともあった。 324   5.6134 0.0636  1.0000   0.1355
は周囲の確認を待つことも 195   6.5625 0.0878  1.0000   0.0616
の働きかけを待つこともあ 461   4.0872 0.1178  1.0000   0.1886
では周囲の確認を待つこと 163   6.4951 0.1233  1.0000   0.0512
周囲の確認を待つこともあ 203   5.5169 0.1464  1.0000   0.0639


入社時メモ : 503 個のn-gramを検定
  Bonferroni補正後 p<0.05 の数: 65
        ngram   n   gap_pt   p  p_bonf  prev_te
リング職として入社。\n・ 158 -26.9973 0.0     0.0   0.0532
アリング職として入社。\n 158 -26.9973 0.0     0.0   0.0532
グ職として入社。\n・人物 154 -26.7905 0.0     0.0   0.0524
ング職として入社。\n・人 154 -26.7905 0.0     0.0   0.0524
 ・エンジニアリング職とし 166 -24.8251 0.0     0.0   0.0576
 T・エンジニアリング職と 166 -24.8251 0.0     0.0   0.0576


### 結果

- **上司からのフィードバック / 同僚からのフィードバック: Bonferroni補正後に有意なn-gramはゼロ**。
  v4の手選びキーワードによるnull結果が、総当たりでも再現した。この2列にはテキストマイニングで
  取り出せる定着シグナルは無いと結論してよい。
- **入社時メモ: 65個が有意**だが、内訳は全て「IT・エンジニアリング職として入社」（＝`初期職種`）と
  「在宅勤務を必須条件としていない」（＝ブロックEの在宅希望フラグ）で、既存の構造化列・既存ブロックと
  完全に重複している。メモからの新規シグナル抽出も枯渇。

## 8. 交絡チェック（見かけ上有望な3候補）

単純集計では有望に見えるが、既存の強い変数で層別すると消える候補を3つ検証する
（v5の教訓「単純集計で大きく見える差は、まず主要な既存特徴量で層別する」の実践）。

In [8]:
# (a) 上司ID のランダム効果
m0=monthly_tr[monthly_tr.経過月数==0].set_index("社員ID")["上司ID"]
P["mgr"]=m0.reindex(P.index)
gm=P.groupby("mgr")[TARGET_COL].agg(["size","mean"]); gm=gm[gm["size"]>=3]
p0=y.mean()
obs=np.average((gm["mean"]-p0)**2,weights=gm["size"]); exp=np.average(p0*(1-p0)/gm["size"],weights=gm["size"])
print("(a) 上司ランダム効果: 観測分散%.5f / 二項ノイズ期待%.5f = %.2f倍" % (obs,exp,obs/exp))
sub=P[P.mgr.notna()].copy(); ag=sub.groupby("mgr")[TARGET_COL].agg(["sum","count"])
sub["loo"]=(sub.mgr.map(ag["sum"])-sub[TARGET_COL])/(sub.mgr.map(ag["count"])-1)
s2=sub[sub.mgr.map(ag["count"])>=3]
print("    LOO上司平均 vs 本人ラベル corr=%.4f" % s2.loo.corr(s2[TARGET_COL]))

# (b) 同期入社人数
coh=P.groupby("入社日").size(); P["cohort_size"]=P["入社日"].map(coh)
print("\n(b) 同期人数: 全体corr=%.4f" % P.cohort_size.corr(y))
for g,s in P.groupby("入社区分"): print("    入社区分=%s内: corr=%.4f (n=%d)" % (g, s.cohort_size.corr(s[TARGET_COL]), len(s)))
apr=P[P["入社日"].dt.month==4]; print("    4月入社内: corr=%.4f (n=%d)" % (apr.cohort_size.corr(apr[TARGET_COL]), len(apr)))

# (c) 自己学習テーマの「可搬性」指標
PORTABLE=["Pythonプログラミング基礎","システム設計基礎","SQLによるデータ操作","BIダッシュボード作成",
          "データ可視化実践","統計学基礎","機械学習入門","クラウド基盤入門","アジャイル開発実践",
          "Pythonによるデータ分析","データリテラシー基礎","情報セキュリティ基礎"]
d=monthly_tr[monthly_tr["自己学習（詳細）"]!="受講なし"].copy()
d["theme"]=d["自己学習（詳細）"].str.extract(r"^([^0-9（(]+)")[0].str.strip().str.rstrip("：")
d["port"]=d.theme.isin(PORTABLE)
gg=d.groupby("社員ID").agg(n=("theme","size"),np_=("port","sum")); gg["share"]=gg.np_/gg.n
P["port_share"]=gg["share"].reindex(P.index)
print("\n(c) 自己学習の可搬スキル比率: 全体corr=%.4f" % P.port_share.corr(y))
rows=[]
for job,s in P[P.port_share.notna()].groupby("初期職種"):
    rows.append((job,len(s),round(s.port_share.corr(s[TARGET_COL]),4)))
RR=pd.DataFrame(rows,columns=["初期職種","n","corr"]); print(RR.to_string(index=False))
print("    職種内プール相関(Fisher-z): %.4f" % np.tanh(np.average(np.arctanh(RR["corr"]),weights=RR.n-3)))

(a) 上司ランダム効果: 観測分散0.07854 / 二項ノイズ期待0.07034 = 1.12倍
    LOO上司平均 vs 本人ラベル corr=0.0717

(b) 同期人数: 全体corr=-0.1373
    入社区分=中途内: corr=-0.0222 (n=736)
    入社区分=新卒内: corr=-0.0088 (n=2025)
    4月入社内: corr=-0.0143 (n=2138)

(c) 自己学習の可搬スキル比率: 全体corr=-0.1450
             初期職種   n    corr
      IT・エンジニアリング 593  0.0867
           コーポレート 397 -0.1204
データ・商品企画・コンサルティング 267  0.0354
  リスク・金融・コンプライアンス 368 -0.1025
          営業・顧客対応 596 -0.1013
             業務運用 537  0.0097
    職種内プール相関(Fisher-z): -0.0291


### 結果 — 3件とも交絡（不採用）

| 候補 | 単純集計 | 層別/ノイズ補正後 | 判定 |
|---|---|---|---|
| 上司IDのランダム効果 | LOO相関 0.072 | 二項ノイズの1.12倍しかない。部署(0.109)より弱く、その部署はv5で「職種の再パッケージ」と判明済み | 不採用 |
| 同期入社人数 | corr -0.137 | 入社区分内 -0.02/-0.01、4月入社内 -0.014 | 新卒4月一括採用の代理変数、不採用 |
| 自己学習の可搬スキル比率 | corr -0.145、五分位で 0.648→0.460 と単調 | 職種内プール相関 -0.029、職種により符号反転 | `初期職種`の代理変数、不採用 |

(c)は特に紛らわしい: 「可搬的な技術スキル（Python/SQL/クラウド）を学ぶ人ほど辞める」という
解釈しやすいストーリーと綺麗な単調性を持つが、実体は「IT・エンジニアリング職の人がPythonを学び、
IT職の定着率が低い」という既知の情報でしかなかった。

## 9. まとめ

詳細は `report/md/data_exploration_v6_report.md` を参照。